# Phase 7: SHAP Explainability

This notebook loads the best-performing Random Forest model and uses SHAP (Shapley Additive Explanations) to explain feature contributions. It replicates the global multi-class feature importance stacked bar chart (Figure 6) and generates beeswarm summary plots for all 12 lithofacies classes (Figure 7 panels a-l).

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../'))
from src.models import load_models
from src.explain import compute_shap_explanations, plot_global_importance, plot_per_class_beeswarm

# 1. Load test data and the best model
df = pd.read_parquet('../data/interim/processed_features.parquet')
wells = df['WELL_ID'].unique()
df_test = df[df['WELL_ID'].isin(wells[8:])]

wavelet_cols = ['DEPTH_MD', 'CALI', 'RSHA', 'RMED', 'RDEP', 'RHOB', 'GR', 'NPHI', 'PEF', 'DTC', 'SP', 'BS'] \
               + [f'{col}_CWT' for col in ['GR', 'NPHI', 'SP', 'RDEP', 'RHOB', 'DTC', 'PEF']]

models_wav = load_models('wavelet')
rf_model = models_wav['Random Forest']
print('Model and data successfully loaded.')

## 1. Compute SHAP Values

We initialize a TreeExplainer and calculate SHAP contributions on 200 representative test samples to optimize execution speed.

In [ ]:
explainer, shap_values, X_sample = compute_shap_explanations(rf_model, df_test[wavelet_cols], num_samples=200)
print('SHAP values computed.')

## 2. Replicating Global Feature Importance Stacked Bar Chart

Visualizes how each of the 19 features contributes across all 12 classes (replicating Figure 6 from the paper).

In [ ]:
lithology_labels = [
    'Sandstone', 'Sandstone/Shale', 'Shale', 'Marl', 'Dolomite', 'Limestone',
    'Chalk', 'Halite', 'Anhydrite', 'Tuff', 'Coal', 'Basement'
]

plot_global_importance(shap_values, X_sample, lithology_labels, '../plots/shap_global_importance.png')

# Display generated image
from IPython.display import Image
Image(filename='../plots/shap_global_importance.png')

## 3. Replicating Per-Class Beeswarm Plots

Generates 12 beeswarm plots (replicating panels a-l in Figure 7) showing feature value effects on prediction score.

In [ ]:
plot_per_class_beeswarm(shap_values, X_sample, lithology_labels, '../plots')
print('All 12 beeswarm plots saved in the plots/ folder!')